# Multiscale U-Net (ours)

The same multiscale cold-diffusion framework as notebook 09, with a time-conditioned ConvNeXt U-Net
(dim 128, multipliers (1, 2, 4, 8), 224 M parameters) as the restoration network.
Weights: `weights/multiscale_unet.pth` (199k updates).

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
The forward model is the multiscale operator $A$: 40 successive upward continuations of 20 m (800 m in
total), each with zero padding to 36 × 36 and cropping back to 32 × 32. The observation is
$y = x_{40} = A^{40} x_0$.

In [2]:
from gravdc import MultiscaleOperator
from gravdc.sampling import multiscale_dc

names, x = load_test_set(DATA_DIR)
x = x.to(device)
A = MultiscaleOperator(steps=40, step=20.0, device=device)
y = A(x)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference
`multiscale_dc` runs the restoration network $R(x_t, t)$ from $t = 40$ down to $1$:

* **Single-shot**: $\hat x_0 = R(y, 40)$, one network evaluation.
* **Stabilized DC**: $x_{t-1} = x_t - A^{t}R(x_t,t) + A^{t-1}R(x_t,t)$.
* **Recursive DC**: $x_{t-1} = A^{t-1}R(x_t,t)$.

In [3]:
model = load_model("multiscale_unet", device)
recs = {routine: multiscale_dc(model, y, A, routine) for routine in ['single-shot', 'stabilized', 'recursive']}

## Metrics

In [4]:
LABEL = {"single-shot": "Single-shot", "stabilized": "Stabilized DC", "recursive": "Recursive DC"}
for routine, x_hat in recs.items():
    row = f"Multiscale U-Net ({LABEL[routine]})"
    per_image = evaluate(x, x_hat)
    compare(row, summarize(per_image))
    save_result(row, per_image, names, RESULTS_DIR)

Multiscale U-Net (Single-shot)
  metric             this run              Table 3
  PSNR       41.8093 ± 3.0735     41.8093 ± 3.0735  ✓
  SSIM        0.9953 ± 0.0069      0.9953 ± 0.0069  ✓
  RE          0.0427 ± 0.0186      0.0427 ± 0.0186  ✓
  RMSE        0.0152 ± 0.0078      0.0152 ± 0.0078  ✓
Multiscale U-Net (Stabilized DC)
  metric             this run              Table 3
  PSNR       40.8483 ± 3.0420     40.8483 ± 3.0420  ✓
  SSIM        0.9943 ± 0.0078      0.9943 ± 0.0078  ✓
  RE          0.0478 ± 0.0203      0.0478 ± 0.0203  ✓
  RMSE        0.0169 ± 0.0085      0.0169 ± 0.0085  ✓
Multiscale U-Net (Recursive DC)
  metric             this run              Table 3
  PSNR       28.6774 ± 2.1266     28.6774 ± 2.1266  ✓
  SSIM        0.9499 ± 0.0158      0.9499 ± 0.0158  ✓
  RE          0.1836 ± 0.0362      0.1836 ± 0.0362  ✓
  RMSE        0.0641 ± 0.0156      0.0641 ± 0.0156  ✓


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {name: rec[k] for name, rec in recs.items()}, title=names[k])